# Notebook ٤: مفاهيم Machine Learning على داتا الكورة

Regression وClassification وoverfitting وcross-validation على داتا FPL الحقيقية. scikit-learn، كل خلية اتشغلت فعلًا.

## ١. نجهز الداتا: features وlabel

X هي المدخلات (features) وy هي اللي عايزين نتوقعه (label).

In [ ]:
# مكتبات القراءة والحساب والجداول
import json, numpy as np, pandas as pd
# نقرأ ملف FPL
raw = json.load(open("fpl.json"))
# جدول اللاعبين
df = pd.DataFrame(raw["elements"])
# اسم المركز
df["pos"] = df["element_type"].map({1: "GK", 2: "DEF", 3: "MID", 4: "FWD"})
# السعر الحقيقي
df["price"] = df["now_cost"] / 10
# نبقي على اللي لعب 300 دقيقة فأكتر
df = df[df["minutes"] >= 300].copy()
# أعمدة الـ features
feats = ["minutes", "goals_scored", "assists", "clean_sheets", "bonus", "price"]
# المدخلات والهدف (النقاط)
X, y = df[feats], df["total_points"]
# أحجامهم
print(X.shape, y.shape)
# أول 3 صفوف
X.head(3)


## ٢. Train/Test split وbaseline

لازم نقيس على داتا الموديل ما شافهاش، ونقارن بموديل غبي (baseline).

In [ ]:
# دالة التقسيم
from sklearn.model_selection import train_test_split
# موديل بيرجع المتوسط دايمًا
from sklearn.dummy import DummyRegressor
# مقاييس الخطأ
from sklearn.metrics import mean_absolute_error, r2_score

# 80% تدريب و20% اختبار
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
# الـ baseline: يتوقع المتوسط
base = DummyRegressor().fit(Xtr, ytr)
# متوسط الخطأ المطلق للـ baseline
print("baseline MAE:", round(mean_absolute_error(yte, base.predict(Xte)), 2))


## ٣. Linear Regression

أبسط موديل: النقاط = مجموع (وزن × feature) + ثابت.

In [ ]:
# الانحدار الخطي
from sklearn.linear_model import LinearRegression
# ندرب الموديل
lr = LinearRegression().fit(Xtr, ytr)
# نتوقع على الاختبار
pred = lr.predict(Xte)
# متوسط الخطأ
print("MAE:", round(mean_absolute_error(yte, pred), 2))
# R2: نسبة التباين اللي الموديل فسّرها
print("R2:", round(r2_score(yte, pred), 3))
# وزن كل feature
print({k: round(float(v), 3) for k, v in zip(feats, lr.coef_)})


In [ ]:
# جدول مقارنة الحقيقي بالمتوقع
res = pd.DataFrame({"player": df.loc[yte.index, "web_name"], "actual": yte, "predicted": pred.round(1)})
# أول 6 لاعبين
res.head(6)


## ٤. Overfitting

موديل بيحفظ التدريب ويفشل في الجديد. بنشوفه بمقارنة خطأ التدريب والاختبار.

In [ ]:
# شجرة قرار
from sklearn.tree import DecisionTreeRegressor
# أعماق مختلفة للشجرة
for depth in [1, 3, 6, 12, None]:
# ندرب شجرة بالعمق ده
    t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(Xtr, ytr)
# خطأ التدريب
    tr = mean_absolute_error(ytr, t.predict(Xtr))
# خطأ الاختبار
    te = mean_absolute_error(yte, t.predict(Xte))
# نطبع المقارنة
    print(f"depth={str(depth):4} train MAE={tr:5.2f}  test MAE={te:5.2f}")


الملاحظة: خطأ التدريب نزل لـ 0 لكن خطأ الاختبار فضل حوالي 5.7. الفجوة دي هي الـ overfitting. في الداتا الصغيرة دي خطأ الاختبار ما زادش، بس الفجوة بتقول إن الموديل بيحفظ.

## ٥. Cross-validation وRandom Forest

تقسيمة واحدة ممكن تضلل، فنقسم 5 مرات. Random Forest بيجمع أشجار كتير.

In [ ]:
# غابة عشوائية
from sklearn.ensemble import RandomForestRegressor
# حساب الدرجة بالتقسيم المتكرر
from sklearn.model_selection import cross_val_score
# الموديل
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=0)
# نقارن الاتنين
for name, m in [("linear", LinearRegression()), ("forest", rf)]:
# MAE بالسالب لكل تقسيمة
    s = cross_val_score(m, X, y, cv=5, scoring="neg_mean_absolute_error")
# متوسط الخطأ
    print(name, "CV MAE:", round(-s.mean(), 2))


In [ ]:
# ندرب الغابة على التدريب
rf.fit(Xtr, ytr)
# أهمية كل feature
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
# نعرضها مرتبة
imp.round(3)


## ٦. Classification: نتوقع مركز اللاعب

هنا الـ label فئة (DEF/MID/FWD) مش رقم.

In [ ]:
# توحيد المقاييس
from sklearn.preprocessing import StandardScaler
# ربط الخطوات
from sklearn.pipeline import make_pipeline
# الموديل
from sklearn.linear_model import LogisticRegression
# نستبعد حراس المرمى
d = df[df["pos"].isin(["DEF", "MID", "FWD"])].copy()
# أهداف لكل 90 دقيقة
d["g90"] = d["goals_scored"] / d["minutes"] * 90
# تمريرات حاسمة لكل 90
d["a90"] = d["assists"] / d["minutes"] * 90
# نظافة شباك لكل 90
d["cs90"] = d["clean_sheets"] / d["minutes"] * 90
# نقاط البونص لكل 90
d["bonus90"] = d["bonus"] / d["minutes"] * 90
# الـ features الجديدة
f2 = ["g90", "a90", "cs90", "bonus90", "price"]
# تقسيم بنفس نسب المراكز
A_tr, A_te, b_tr, b_te = train_test_split(d[f2], d["pos"], test_size=0.25, random_state=1, stratify=d["pos"])
# نوحد المقاييس ثم نصنف
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# تدريب
clf.fit(A_tr, b_tr)
# الدقة على الاختبار
print("accuracy:", round(clf.score(A_te, b_te), 3))


In [ ]:
# مصفوفة الخلط والتقرير
from sklearn.metrics import confusion_matrix, classification_report
# توقعات الاختبار
p = clf.predict(A_te)
# ترتيب الفئات
labs = ["DEF", "MID", "FWD"]
# الصفوف الحقيقي والأعمدة المتوقع
print(pd.DataFrame(confusion_matrix(b_te, p, labels=labs), index=labs, columns=labs))
# precision وrecall وf1
print(classification_report(b_te, p, labels=labs, zero_division=0))


كل اللي ظهر هنا حقيقي: الموديل ما عرفش يتوقع ولا مهاجم صح (6 مهاجمين بس في الاختبار، يعني الفئة قليلة). ده مثال على class imbalance، ودقة 0.65 لوحدها بتخبي ده.